In [1]:
!git clone https://github.com/Nusultan11/imdb-sentiment.git
%cd /content/imdb-sentiment

fatal: destination path 'imdb-sentiment' already exists and is not an empty directory.
/content/imdb-sentiment


In [2]:
%pip install -e .

Obtaining file:///content/imdb-sentiment
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for imdb-sentiment (pyproject.toml) ... done
  Created wheel for imdb-sentiment: filename=imdb_sentiment-0.1.0-0.editable-py3-none-any.whl size=5195 sha256=70d9ecf9666c262821704322f53c158cc54f5e4b56797510aa51dce471053fab
  Stored in directory: /tmp/pip-ephem-wheel-cache-qmavvn66/wheels/3e/08/09/1f8f2dc306b20c0f6d4e5a4ddb080c355c7ce988d70820356f
Successfully built imdb-sentiment
  Attempting uninstall: imdb-sentiment
    Found existing installation: imdb-sentiment 0.1.0
    Uninstalling imdb-sentiment-0.1.0:
      Successfully uninstalled imdb-sentiment-0.1.0


In [12]:
import sys
from pathlib import Path
import json
import shutil

from imdb_sentiment.settings import load_config, LSTMModelConfig

from google.colab import files

In [4]:
project_root = Path("/content/imdb-sentiment").resolve()
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

In [5]:
import imdb_sentiment
print("Project import: OK")

Project import: OK


In [6]:
config_path = Path("/content/imdb-sentiment/configs/experiments/lstm_baseline_v1.yaml")
config = load_config(config_path)

print("Config path:", config_path)
print("Family:", config.experiment.family)
print("Name:", config.experiment.name)
print("Model type:", config.model.type)

assert isinstance(config.model, LSTMModelConfig)

print("\n=== LSTM CONFIG CHECK ===")
print("vocab_size:", config.model.vocab_size)
print("max_length:", config.model.max_length)
print("embedding_dim:", config.model.embedding_dim)
print("hidden_dim:", config.model.hidden_dim)
print("batch_size:", config.model.batch_size)
print("epochs:", config.model.epochs)
print("dropout:", config.model.dropout)
print("lr:", config.model.lr)

Config path: /content/imdb-sentiment/configs/experiments/lstm_baseline_v1.yaml
Family: lstm
Name: baseline_v1
Model type: lstm

=== LSTM CONFIG CHECK ===
vocab_size: 30000
max_length: 512
embedding_dim: 128
hidden_dim: 128
batch_size: 32
epochs: 5
dropout: 0.3
lr: 0.001


In [7]:
assert config.experiment.family == "lstm"
assert config.model.type == "lstm"
assert config.model.vocab_size == 30000, f"Expected vocab_size=30000, got {config.model.vocab_size}"
assert config.model.max_length == 512, f"Expected max_length=512, got {config.model.max_length}"

print("LSTM baseline config is ready.")

LSTM baseline config is ready.


In [8]:
!python -m imdb_sentiment.cli train --config configs/experiments/lstm_baseline_v1.yaml

Training finished.
Accuracy: 0.8408
Precision: 0.8565
Recall: 0.8196
F1: 0.8377
Model saved to: /content/imdb-sentiment/artifacts/experiments/lstm/baseline_v1/model.pt
Validation metrics saved to: /content/imdb-sentiment/artifacts/experiments/lstm/baseline_v1/val_metrics.json


In [10]:
lstm_val_path = Path("/content/imdb-sentiment/artifacts/experiments/lstm/baseline_v1/val_metrics.json")
lstm_val_metrics = json.loads(lstm_val_path.read_text(encoding="utf-8"))

print(lstm_val_metrics)

{'loss': 0.39757923155453556, 'accuracy': 0.8408, 'precision': 0.8565471226021685, 'recall': 0.819632881085395, 'f1': 0.8376835236541599}


In [11]:
!ls -R /content/imdb-sentiment/artifacts/experiments/lstm/baseline_v1

/content/imdb-sentiment/artifacts/experiments/lstm/baseline_v1:
model.pt	      training_history.json  vocab.json
training_config.json  val_metrics.json


In [13]:
src_dir = Path("/content/imdb-sentiment/artifacts/experiments/lstm/baseline_v1")
package_dir = Path("/content/lstm_baseline_v1_download")

if package_dir.exists():
    shutil.rmtree(package_dir)

package_dir.mkdir(parents=True, exist_ok=True)

shutil.copytree(src_dir, package_dir / "baseline_v1", dirs_exist_ok=True)

config_src = Path("/content/imdb-sentiment/configs/experiments/lstm_baseline_v1.yaml")
shutil.copy2(config_src, package_dir / "lstm_baseline_v1.yaml")

archive_path = shutil.make_archive(
    base_name="/content/lstm_baseline_v1_artifacts",
    format="zip",
    root_dir=package_dir,
)

print("Archive created:", archive_path)
files.download(archive_path)

Archive created: /content/lstm_baseline_v1_artifacts.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>